In [1]:
import re
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

In [2]:
TARGET_RECALL = 0.97
GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)

In [3]:
df = pd.read_excel("/content/casenum.xlsx")
X = df["Решение кейса"].astype(str).str.replace(
    r"<br\s*/?>", " ", regex=True)
y = df["Кейс"]

In [4]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 1), min_df=3,
                              sublinear_tf=True)),
    ("clf", LogisticRegressionCV(Cs=[1, 3, 10, 30], cv=3,
                                 class_weight="balanced",
                                 max_iter=2000, n_jobs=-1)),
])
proba = cross_val_predict(
    pipe, X, y,
    cv=StratifiedKFold(5, shuffle=True, random_state=0),
    method="predict_proba")
pipe.fit(X, y)
joblib.dump(pipe, "case_classifier.joblib")
classes = pipe.classes_

In [5]:
df_proba = df[["Кейс", "Решение кейса"]].copy()
df_proba["target"] = y
df_proba["proba_target"] = proba[
    np.arange(len(y)), np.searchsorted(classes, y)]
for k, c in enumerate(classes):
    df_proba[f"proba_case_{c}"] = proba[:, k]
df_proba.to_excel("case_proba.xlsx", index=False)

In [6]:
# Подбор порога
rows = []
for c in sorted(df_proba["target"].unique()):
    is_pos = (df_proba["target"] == c).values
    score = df_proba[f"proba_case_{c}"].values
    best = None
    for t in GRID:
        pred = score >= t
        recall = pred[is_pos].mean()
        if recall >= TARGET_RECALL:
            best = t
    if best is None:
        best = GRID[0]
    rows.append({"case": int(c), "threshold": best})
print(pd.DataFrame(rows).to_string(index=False))

 case  threshold
    1       0.39
    2       0.03
    3       0.05
    4       0.13
    5       0.22
    6       0.05
    7       0.27
    8       0.01
    9       0.12
   10       0.19


In [7]:
text_right_case = (
    "Я выбрал отрасль доставки готовой еды. ЦА — малые кухни и частные "
    "повара, работающие на маркетплейсах. Им нужны деньги на упаковку и "
    "продукты, а также контроль чаевых курьеров. Я предлагаю "
    "экосистемный продукт «Альфа.Еда» из пяти модулей. Модуль 1 — "
    "кредит на закупку продуктов с отсрочкой 30 дней и автоматическим "
    "списанием по факту приготовления каждого блюда через интеграцию с "
    "кассой и системой учёта. Модуль 2 — умный складской учёт, который "
    "сам заказывает у поставщиков при снижении остатков (партнёр "
    "«МойСклад»), чтобы повар не тратил время на заказы и не держал "
    "лишний запас. Модуль 3 — система контроля чаевых курьеров через "
    "GPS-трекинг и фотофиксацию заказа у двери, чтобы курьеры не "
    "округляли чаевые в свою пользу (партнёр «ГлонассСофт»), потери "
    "выручки снижаются с 10% до 1%. Модуль 4 — страховка от брака и "
    "возвратов (партнёр «АльфаСтрахование»), если клиент вернул еду, "
    "страховка компенсирует 80% себестоимости. Модуль 5 — агрегатор "
    "поставщиков продуктов с фиксированными ценами на неделю (партнёры "
    "«Агроторг», «Мираторг»), чтобы повар мог планировать закупки без "
    "сюрпризов по ценам. Отличие от конкурентов в комплексном подходе: "
    "ни у Сбера, ни у Т-Банка нет интеграции с поставщиками и контроля "
    "чаевых для доставки. По финансам: разработка 18 млн, интеграции "
    "6 млн. Прогноз: 400 клиентов, доход 30 тыс., выручка 12 млн. "
    "Окупаемость 2,3 года. ЦА сегментирована слабо, риски без "
    "вероятностей, доказательства слабые."
)
wrong_text = (
    "Я выбрал сферу инклюзивного образования, потому что по данным "
    "Минпросвещения в России более 1,5 млн детей с ОВЗ, и 40% из них "
    "не получают качественного образования из-за нехватки "
    "специалистов. Целевая аудитория делится на четыре группы: дети с "
    "нарушениями слуха, дети с нарушениями зрения, дети с РАС "
    "(расстройства аутистического спектра), дети с нарушениями "
    "опорно-двигательного аппарата. Для каждой группы описаны "
    "специфические боли: глухим нужны сурдопереводчики, слепым — "
    "тактильные материалы, детям с РАС — сенсорная разгрузка, детям с "
    "ДЦП — адаптированные тренажёры. Я продумал CJM: родитель узнаёт "
    "о школе → проходит диагностику → получает индивидуальный план → "
    "ребёнок учится → выпускник получает профессию. Решение: создать "
    "\"Альфа-Инклюзия\" — школу с ИИ-ассистентом, который переводит "
    "речь в текст для глухих, описывает изображения для слепых, "
    "регулирует сенсорную нагрузку для детей с РАС и адаптирует "
    "упражнения для детей с ДЦП. Уникальность: мы объединяем все "
    "четыре группы в одном пространстве, но с индивидуальными "
    "траекториями. Партнёры: фонд \"Обнажённые сердца\", \"Яндекс\" "
    "(речевые технологии), МГППУ (методология). Финансы: разработка "
    "ИИ — 40 млн, оборудование — 30 млн, аренда — 10 млн в год, "
    "специалисты — 20 млн в год. 150 учеников, чек — 200 тыс. "
    "Выручка — 30 млн. Дополнительно: гранты — 20 млн, корпоративные "
    "спонсоры — 10 млн. CAC — 25 тыс., LTV — 1,2 млн, NPV — 15 млн, "
    "окупаемость — 3 года. Подушка безопасности — 15% бюджета. "
    "Риски: этические (ИИ может ошибаться в переводе жестов) — "
    "решение: двойная проверка сурдопереводчиком; регуляторные "
    "(лицензирование) — решение: работа с Минпросвещения; кадровые "
    "(нехватка специалистов) — решение: программа переподготовки. "
    "Доказательства: собственный пилот с 20 детьми (улучшение "
    "результатов на 35%), интервью с 15 родителями, A/B-тест с "
    "ИИ-ассистентом и без него."
)

In [8]:
# Проверка
THRESHOLD = 0.10
THRESHOLD_BY_CASE = {2: 0.03, 3: 0.05, 6: 0.05, 8: 0.01}
model_correct_case = joblib.load("case_classifier.joblib")
CASE_IDS = {
    ("ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ МАЛОГО И "
     "СРЕДНЕГО БИЗНЕСА"): 1,
    ("ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ ОРГАНИЗАЦИЙ "
     "КУЛЬТУРНО-ДОСУГОВОГО НАЗНАЧЕНИЯ"): 2,
    ("ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ РАЗВИТИЯ "
     "МАССОВОГО СПОРТА И ФИЗИЧЕСКОЙ КУЛЬТУРЫ"): 3,
    ("КОМПЛЕКСНОЕ ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ СИСТЕМЫ "
     "ОБРАЗОВАНИЯ НА БАЗЕ ИСКУССТВЕННОГО ИНТЕЛЛЕКТА"): 4,
    "ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ НЕФТЕГАЗОВОГО СЕКТОРА": 5,
    ("ФИДЖИТАЛ-ОФИС 2.0 — НОВОЕ ПОКОЛЕНИЕ БАНКОВСКИХ ОТДЕЛЕНИЙ "
     "ДЛЯ ГОРОДОВ С НАСЕЛЕНИЕМ ОТ 30 ТЫСЯЧ ЧЕЛОВЕК"): 6,
    ("ЦЕНТР ПОДГОТОВКИ ПРОФЕССИОНАЛОВ БУДУЩЕГО — "
     "ОБРАЗОВАТЕЛЬНОЕ УЧРЕЖДЕНИЕ 2050"): 7,
    ("ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ РАЗВИТИЯ "
     "ПРОФЕССИОНАЛЬНОГО СПОРТА В РЕГИОНЕ"): 8,
    "ЭЛЕКТРОМОБИЛЬНЫЙ КЛАСТЕР ДЛЯ РЕГИОНА": 9,
    "АНТИКРИЗИСНАЯ СТРАТЕГИЯ ДЛЯ СЕТИ РЕСТОРАНОВ": 10,
}
def check_case(text, case_name):
    # Возвращает 1, если решение подходит к выбранному кейсу, иначе 0
    case_id = CASE_IDS[case_name]
    text = re.sub(r"<br\s*/?>", " ", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    proba = model_correct_case.predict_proba([text])[0]
    class_index = list(model_correct_case.classes_).index(case_id)
    threshold = THRESHOLD_BY_CASE.get(case_id, THRESHOLD)
    return int(proba[class_index] >= threshold)
case_name = ("ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ МАЛОГО И "
             "СРЕДНЕГО БИЗНЕСА")
check_case(wrong_text, case_name)

0